# 2.5 Comprobaciones Segmentacion HSI

Diagnostico de por que `2_Segmentacion_HSI.ipynb` funciona para algunas HSI y falla para otras.

Este notebook no modifica el original. Comprueba lectura ENVI, presencia de NaN y el efecto de la normalizacion usada en `2_Segmentacion_HSI.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
import numpy as np

plt.rcParams['figure.dpi'] = 120


In [ ]:
SPECIMENS = [
    ('SB012', Path(r'Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm_edu.hdr')),
    ('SB013', Path(r'Datos\SB013\SB013\SB013_001\SB013_nrm_edu.hdr')),
    ('SB017', Path(r'Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_nrm_edu.hdr')),
    ('SB018', Path(r'Datos\SB018\SB018\SB018_001\SB018_nrm_edu.hdr')),
    ('SB019', Path(r'Datos\SB019\SB019\SB019_001\SB019_nrm_edu.hdr')),
    ('SB020', Path(r'Datos\SB020\SB020\SB020_001\SB020_nrm_edu.hdr')),
]

for sid, path in SPECIMENS:
    print(f'{sid}: exists={path.exists()} | {path}')


In [ ]:
def parse_envi_header(hdr_path):
    text = Path(hdr_path).read_text(encoding='utf-8', errors='ignore')
    metadata = {}
    current_key = None
    current_value = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.upper() == 'ENVI':
            continue

        if current_key is not None:
            current_value.append(line)
            if '}' in line:
                metadata[current_key] = ' '.join(current_value)
                current_key = None
                current_value = []
            continue

        if '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()
        if value.startswith('{') and '}' not in value:
            current_key = key
            current_value = [value]
        else:
            metadata[key] = value

    return metadata


def parse_wavelengths(metadata):
    values = metadata.get('wavelength')
    if values is None:
        return None
    values = values.strip().strip('{}')
    return np.array([float(v.strip()) for v in values.split(',') if v.strip()], dtype=np.float32)


def envi_dtype(metadata):
    data_type = int(metadata['data type'])
    endian = '<' if int(metadata.get('byte order', 0)) == 0 else '>'
    dtype_map = {1: 'u1', 2: 'i2', 3: 'i4', 4: 'f4', 5: 'f8', 12: 'u2', 13: 'u4', 14: 'i8', 15: 'u8'}
    return np.dtype(endian + dtype_map[data_type])


def find_envi_data_path(hdr_path, metadata):
    hdr_path = Path(hdr_path)
    candidates = []
    for key in ('data file', 'file name', 'image'):
        if key in metadata:
            value = metadata[key].strip().strip('{}').strip('"')
            candidates.append(hdr_path.parent / value)
    candidates.extend([hdr_path.with_suffix(''), hdr_path.parent / hdr_path.stem.replace('_edu', '')])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'No se encontro el binario ENVI asociado a {hdr_path.name}')


def read_envi_band(hdr_path, metadata, band_idx):
    data_path = find_envi_data_path(hdr_path, metadata)
    samples = int(metadata['samples'])
    lines = int(metadata['lines'])
    bands = int(metadata['bands'])
    offset = int(metadata.get('header offset', 0))
    interleave = metadata.get('interleave', 'bsq').strip().lower()
    dtype = envi_dtype(metadata)

    if interleave == 'bsq':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(bands, lines, samples))
        return np.asarray(cube[band_idx], dtype=np.float32)
    if interleave == 'bil':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, bands, samples))
        return np.asarray(cube[:, band_idx, :], dtype=np.float32)
    if interleave == 'bip':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, samples, bands))
        return np.asarray(cube[:, :, band_idx], dtype=np.float32)
    raise ValueError(f'Interleave no soportado: {interleave}')


def nearest_band(wavelengths, target_nm):
    idx = int(np.argmin(np.abs(wavelengths - target_nm)))
    return idx, float(wavelengths[idx])


In [ ]:
def old_robust_normalize(channel, p_low=2, p_high=98):
    """Misma normalizacion que usa 2_Segmentacion_HSI.ipynb."""
    ch = channel.astype(np.float32)
    lo = np.percentile(ch, p_low)
    hi = np.percentile(ch, p_high)
    if hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)
    ch = (ch - lo) / (hi - lo)
    return np.clip(ch, 0, 1)


def nan_safe_robust_normalize(channel, p_low=2, p_high=98):
    """Version segura: ignora NaN e infinitos."""
    ch = channel.astype(np.float32)
    if not np.any(np.isfinite(ch)):
        return np.zeros_like(ch, dtype=np.float32)
    lo = np.nanpercentile(ch, p_low)
    hi = np.nanpercentile(ch, p_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)
    out = np.clip((ch - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def make_pseudo_rgb(hdr_path, normalize_func):
    metadata = parse_envi_header(hdr_path)
    wavelengths = parse_wavelengths(metadata)
    r_idx, r_nm = nearest_band(wavelengths, 650)
    g_idx, g_nm = nearest_band(wavelengths, 550)
    b_idx, b_nm = nearest_band(wavelengths, 450)
    bands = [read_envi_band(hdr_path, metadata, idx) for idx in (r_idx, g_idx, b_idx)]
    rgb = np.stack([normalize_func(band) for band in bands], axis=-1)
    return rgb, {'r_idx': r_idx, 'r_nm': r_nm, 'g_idx': g_idx, 'g_nm': g_nm, 'b_idx': b_idx, 'b_nm': b_nm}, bands, metadata


def count_old_blue_candidates(rgb_float):
    """Aproximacion sin cv2 del umbral usado en 2_Segmentacion_HSI.ipynb."""
    rgb_u8 = (np.clip(rgb_float, 0, 1) * 255).astype(np.uint8)
    rgb01 = rgb_u8.astype(np.float32) / 255.0
    hsv = rgb_to_hsv(rgb01)
    h_cv = hsv[:, :, 0] * 179.0
    s_cv = hsv[:, :, 1] * 255.0
    v_cv = hsv[:, :, 2] * 255.0
    r = rgb_u8[:, :, 0].astype(np.int16)
    g = rgb_u8[:, :, 1].astype(np.int16)
    b = rgb_u8[:, :, 2].astype(np.int16)
    mask_hsv = (h_cv >= 85) & (h_cv <= 125) & (s_cv >= 10) & (s_cv <= 180) & (v_cv >= 40)
    mask_dominance = ((b - r) > 8) & ((g - r) > 3) & (v_cv > 50)
    return int(np.count_nonzero(mask_hsv & mask_dominance)), rgb_u8


In [ ]:
rows = []

for sid, hdr_path in SPECIMENS:
    rgb_old, info, bands, metadata = make_pseudo_rgb(hdr_path, old_robust_normalize)
    rgb_safe, _, _, _ = make_pseudo_rgb(hdr_path, nan_safe_robust_normalize)
    old_blue_px, rgb_old_u8 = count_old_blue_candidates(rgb_old)
    safe_blue_px, rgb_safe_u8 = count_old_blue_candidates(rgb_safe)
    finite_fracs = [float(np.isfinite(band).mean()) for band in bands]
    nan_fracs = [float(np.isnan(band).mean()) for band in bands]
    rows.append({
        'id': sid,
        'shape': (int(metadata['lines']), int(metadata['samples']), int(metadata['bands'])),
        'data_file': find_envi_data_path(hdr_path, metadata).name,
        'finite_R/G/B': tuple(round(x, 6) for x in finite_fracs),
        'nan_R/G/B': tuple(round(x, 6) for x in nan_fracs),
        'old_rgb_nan_frac': round(float(np.isnan(rgb_old).mean()), 6),
        'old_rgb_max': float(np.nanmax(rgb_old)) if np.any(np.isfinite(rgb_old)) else None,
        'old_blue_candidates_px': old_blue_px,
        'safe_blue_candidates_px': safe_blue_px,
        'old_u8_nonzero_frac': round(float((rgb_old_u8 > 0).mean()), 6),
        'safe_u8_nonzero_frac': round(float((rgb_safe_u8 > 0).mean()), 6),
        'rgb_info': info,
    })

for row in rows:
    print(row)


In [ ]:
for sid, hdr_path in SPECIMENS:
    rgb_old, info, _, _ = make_pseudo_rgb(hdr_path, old_robust_normalize)
    rgb_safe, _, _, _ = make_pseudo_rgb(hdr_path, nan_safe_robust_normalize)
    _, old_u8 = count_old_blue_candidates(rgb_old)
    _, safe_u8 = count_old_blue_candidates(rgb_safe)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    axes[0].imshow(old_u8)
    axes[0].set_title(f'{sid} - normalizacion original')
    axes[0].axis('off')

    axes[1].imshow(safe_u8)
    axes[1].set_title(f'{sid} - normalizacion ignorando NaN')
    axes[1].axis('off')
    plt.show()


## Conclusion

El fallo principal no esta en las rutas ni en la existencia de las HSI: las rutas existen y los binarios asociados se encuentran.

La diferencia es que `SB017`, `SB018`, `SB019` y `SB020` contienen valores `NaN` en las bandas usadas para construir la pseudo-RGB. La funcion `robust_normalize` del notebook original usa `np.percentile`, que no ignora `NaN`. Si el percentil sale `NaN`, toda la pseudo-RGB queda contaminada por `NaN`; al convertirla a `uint8`, esos valores se convierten en negro. Por eso despues la mascara azul queda vacia y aparece `No se detecto la caja azul`.

La solucion tecnica es sustituir esa normalizacion por una version con `np.nanpercentile` y `np.nan_to_num`, y despues usar una deteccion de caja azul mas estricta como la del notebook `Deteccion_caja_azul_HSI.ipynb`.